## Due to changes in the metrics on May 4, please refer to this notebook instead.
## https://www.kaggle.com/code/konbu17/neurogolf-2026-may-6-updated

# NeuroGolf 2026 — Blended 401-Task Submission

**Public LB: 6254.65** (updated 2026-04-27, +10.14 from previous)

**One-click submission** for the **NeuroGolf 2026** competition. Just `Run All` and submit the resulting `submission.zip`.

## 🆕 Latest update — task363 silent-fail fix (+10.14 LB)

`task363` was identified as a **silent-fail** task (passes `train + test + arc-gen` locally but contributes **0 pts** on Kaggle's hidden test). We replaced the public ONNX with a 265/265 verified custom implementation over the full 261 arc-gen examples.

## 🙏 Acknowledgments

This notebook builds on the open NeuroGolf community. Sincere thanks to:

- **[beicicc / neurogolf-6233-36-public-score-open-solution](https://www.kaggle.com/code/beicicc/neurogolf-6233-36-public-score-open-solution)** — current strongest base (selector / output-bank / boolean rewrite style)
- **[beicicc / neurogolf-6202-73-public-score-open-solution](https://www.kaggle.com/code/beicicc/neurogolf-6202-73-public-score-open-solution)** — earlier base
- **[beicicc / neurogolf-6061-74-public-score-open-solution](https://www.kaggle.com/code/beicicc/neurogolf-6061-74-public-score-open-solution)** — earlier base
- **[beicicc / neurogolf-5546-64-public-score-open-solution](https://www.kaggle.com/code/beicicc/neurogolf-5546-64-public-score-open-solution)** — earlier base
- **[afr1ste / neurogolf-5501-78-public-score-open-solution](https://www.kaggle.com/code/afr1ste/neurogolf-5501-78-public-score-open-solution)** — selector-style base
- **[afr1ste / neurogolf-5395-71-public-score-open-solution](https://www.kaggle.com/code/afr1ste/neurogolf-5395-71-public-score-open-solution)** — earlier base
- **[vyankteshdwivedi / neurogolf-multi-source-onnx-solver](https://www.kaggle.com/code/vyankteshdwivedi/neurogolf-multi-source-onnx-solver)** — extreme op-fusion / minimization
- **[jonathanchan / ngc26-constraint-smart-logic-mix-blending](https://www.kaggle.com/code/jonathanchan/ngc26-constraint-smart-logic-mix-blending)** — constraint-smart logic
- **[Grand Master discussion #694628](https://www.kaggle.com/competitions/neurogolf-2026/discussion/694628)** — `reshape(-1, 3)` data-structure hack
- **[thisray / neurogolf-4743-93-submission-task-table](https://www.kaggle.com/datasets/thisray/neurogolf-4743-93-submission-task-table)** — agent-AI driven task exploration
- The **NeuroGolf 2026 starter notebooks** (Michael D. Moffitt, organizers)
- The **Kaggle community at large**

For a **clean / bug-free** version of this submission, see also: [NeuroGolf-2026 Bug-Free Blend [LB 5126]](https://www.kaggle.com/code/konbu17/neurogolf-2026-bug-free-blend-lb-5126)

## 🛠️ Method

This notebook takes the strongest public notebook as a base (currently beicicc 6233.36) and applies a small, leaderboard-verified set of overrides for tasks where a different ONNX scores higher on the leaderboard than the local `onnx_tool` cost would suggest.

The current override set is small — a custom `task322` (3x3 column gravity-fill) implementation plus the new `task363` silent-fail fix. The base notebook ships tiny ONNXs for these tasks that pass local `arc-gen` but fail on hidden test cases (silent fail), so substituting actually-correct ONNXs recovers the leaderboard points.

## 📊 Score evolution (rounded)

| Stage | LB |
|---|---:|
| 3-source blend (afr + vya + ours) | 5283.85 |
| + jonathanchan blend | 5331.49 |
| + GM `task007` reshape-trick + thisray `task184` | 5344.29 |
| + per-task LB-bisection on top of the artemnazemtsev base | 5384.67 |
| + custom `task221` ONNX (tile-grid dispatch) | 5385.72 |
| + adopt new afr1ste 5395 base | 5395.71 |
| + ours / artem trio / GM / dim-scrub / small-gap LB-bisected on afr1ste 5395 | 5419.87 |
| + custom `task370` ONNX (4-direction unit-shift cache, sum 1.40M) | 5420.17 |
| + custom `task158` ONNX (path-only stamp, D4×scale origin map, sum 16.06M) | 5428.58 |
| + adopt new afr1ste 5501 base, keep our custom `task370` / `task158` on top | 5510.48 |
| + adopt new beicicc 5518 base (5 tasks swapped in) | 5517.81 |
| + adopt new beicicc 5546 base (45 tasks swapped in) | 5546.38 |
| + `task378` swap to beicicc 5546 boolean-style version | 5546.64 |
| + custom `task322` ONNX (column gravity-fill, recovers silent fail) | 5557.79 |
| + adopt new beicicc 6061 base, keep our `task322` fix on top | 6072.89 |
| + adopt new beicicc 6202 base, keep our `task322` fix on top | 6213.88 |
| + adopt new beicicc 6233 base, keep our `task322` fix on top | 6244.51 |
| **+ custom `task363` silent-fail fix (this update, shift-based correlation)** | **6254.65 (+10.14)** |


In [ ]:
# Diagnostic: what is attached?
import os, glob
for root, dirs, files in os.walk('/kaggle/input'):
    print(f'DIR: {root} ({len(files)} files, {len(dirs)} subdirs)')
    for f in files[:3]:
        full = os.path.join(root, f)
        print(f'  - {f} ({os.path.getsize(full):,} bytes)')
    if len(files) > 3:
        print(f'  ... and {len(files)-3} more')

In [ ]:
import os, zipfile, glob

# Search for task*.onnx anywhere under /kaggle/input/
all_onnx = glob.glob('/kaggle/input/**/task*.onnx', recursive=True)
print(f'found {len(all_onnx)} task*.onnx files via recursive glob')

# Also try /kaggle/input/*/submission.zip (in case dataset preserved as zip)
zip_candidates = glob.glob('/kaggle/input/**/submission.zip', recursive=True)
print(f'found {len(zip_candidates)} submission.zip files: {zip_candidates}')

DST = '/kaggle/working/submission.zip'

if zip_candidates:
    # Use the pre-built zip directly
    import shutil
    src = max(zip_candidates, key=os.path.getsize)
    shutil.copyfile(src, DST)
    print(f'\nCopied: {src} -> {DST}')
elif all_onnx:
    # Build zip from .onnx files
    with zipfile.ZipFile(DST, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in sorted(all_onnx):
            z.write(f, arcname=os.path.basename(f))
    print(f'\nZipped {len(all_onnx)} files -> {DST}')
else:
    raise RuntimeError('No task*.onnx or submission.zip found in /kaggle/input/. Make sure the konbu17/neurogolf-2026-blended-401-v117 dataset is attached.')

print(f'Output size: {os.path.getsize(DST):,} bytes ({os.path.getsize(DST)/1024/1024:.2f} MB)')
with zipfile.ZipFile(DST) as z:
    names = sorted(z.namelist())
    print(f'Files in zip: {len(names)}')
    print(f'First 3: {names[:3]}')
    print(f'Last 3: {names[-3:]}')

## Submit

Kaggle's **Submit** button picks up `/kaggle/working/submission.zip` automatically. 🎯

## Closing thanks

Thanks again to all the upstream notebook authors and the Kaggle community. If you build on this, please cite the upstream notebooks above. ✨
